In [29]:
# Resolving issues regarding mlxtend library incompatibilities. This may or may not be necessary
# !pip uninstall numpy mlxtend -y
# !pip install numpy==1.26.4
# !pip install mlxtend --no-cache-dir

import pandas as pd
from nba_api.stats.endpoints import leaguedashteamstats
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules
from IPython.display import display

SEASON = ["2024-25"]
SEASON_TYPE = "Regular Season"
MIN_SUPPORT = 0.2
MIN_CONFIDENCE = 0.65

def fetch_team_advanced_stats(season):

    response = leaguedashteamstats.LeagueDashTeamStats(
        season=season,
        season_type_all_star=SEASON_TYPE,
        measure_type_detailed_defense="Advanced",
        per_mode_detailed="PerGame"
    )

    df = response.get_data_frames()[0]

    return df

def create_buckets(df):

    df = df[[
        "TEAM_NAME",
        "OFF_RATING",
        "DEF_RATING",
        "PACE",
        "TS_PCT",
        "W_PCT"
    ]].dropna()

    df["High_Offense"] = df["OFF_RATING"] > df["OFF_RATING"].median()
    df["Elite_Defense"] = df["DEF_RATING"] < df["DEF_RATING"].median()
    df["Fast_Pace"] = df["PACE"] > df["PACE"].median()
    df["Efficient_Shooting"] = df["TS_PCT"] > df["TS_PCT"].median()
    df["Winning_Team"] = df["W_PCT"] > 0.6

    return df

def build_transactions(df):

    traits = [
        "High_Offense",
        "Elite_Defense",
        "Fast_Pace",
        "Efficient_Shooting",
        "Winning_Team"
    ]

    transactions = []

    for _, row in df.iterrows():
        items = [trait for trait in traits if row[trait]]
        transactions.append(items)

    return transactions

def run_apriori(transactions):

    te = TransactionEncoder()
    te_array = te.fit(transactions).transform(transactions)
    df_encoded = pd.DataFrame(te_array, columns=te.columns_)

    frequent_itemsets = apriori(
        df_encoded,
        min_support=MIN_SUPPORT,
        use_colnames=True
    )

    rules = association_rules(
        frequent_itemsets,
        metric="confidence",
        min_threshold=MIN_CONFIDENCE
    )

    return frequent_itemsets, rules

def main():

    print(f"\nFetching {SEASON} NBA Advanced Team Stats...\n")

    all_dfs = []

    for season in SEASON:
        print(f"Fetching season {season}")
        season_df = fetch_team_advanced_stats(season)
        season_df["SEASON"] = season
        all_dfs.append(season_df)
    
    df = pd.concat(all_dfs, ignore_index=True)

    df = create_buckets(df)

    transactions = build_transactions(df)

    frequent_itemsets, rules = run_apriori(transactions)
    rules['antecedents'] =  rules['antecedents'].apply(list)
    rules['consequents'] = rules['consequents'].apply(list)

    print("\nFrequent Itemsets:")
    print(frequent_itemsets.sort_values(by="support", ascending=False))

    print("\nAssociation Rules: Support")
    display(
        rules[['antecedents','consequents','support','confidence','lift']]
        .sort_values(by='support', ascending=False).head(15)
    )

    print("\nAssociation Rules: Confidence")
    display(
        rules[['antecedents','consequents','confidence','support','lift']]
        .sort_values(by='confidence', ascending=False).head(15)
    )

    print("\nAssociation Rules: LIft")
    display(
        rules[['antecedents','consequents', 'lift', 'support','confidence']]
        .sort_values(by='lift', ascending=False).head(15)
    )



if __name__ == "__main__":
    main()


Fetching ['2024-25'] NBA Advanced Team Stats...

Fetching season 2024-25

Frequent Itemsets:
     support                                           itemsets
0   0.500000                    frozenset({Efficient_Shooting})
1   0.500000                         frozenset({Elite_Defense})
2   0.500000                             frozenset({Fast_Pace})
3   0.500000                          frozenset({High_Offense})
7   0.433333      frozenset({High_Offense, Efficient_Shooting})
10  0.366667           frozenset({High_Offense, Elite_Defense})
4   0.300000                          frozenset({Winning_Team})
5   0.300000     frozenset({Elite_Defense, Efficient_Shooting})
13  0.300000            frozenset({High_Offense, Winning_Team})
14  0.300000  frozenset({High_Offense, Elite_Defense, Effici...
17  0.266667  frozenset({Winning_Team, High_Offense, Efficie...
8   0.266667      frozenset({Winning_Team, Efficient_Shooting})
6   0.266667         frozenset({Fast_Pace, Efficient_Shooting})
12  0.2333

,antecedents,consequents,support,confidence,lift
0,[High_Offense],[Efficient_Shooting],0.433333,0.866667,1.733333
1,[Efficient_Shooting],[High_Offense],0.433333,0.866667,1.733333
3,[High_Offense],[Elite_Defense],0.366667,0.733333,1.466667
4,[Elite_Defense],[High_Offense],0.366667,0.733333,1.466667
6,[Winning_Team],[High_Offense],0.300000,1.000000,2.000000
8,"[High_Offense, Efficient_Shooting]",[Elite_Defense],0.300000,0.692308,1.384615
7,"[High_Offense, Elite_Defense]",[Efficient_Shooting],0.300000,0.818182,1.636364
9,"[Elite_Defense, Efficient_Shooting]",[High_Offense],0.300000,1.000000,2.000000
2,[Winning_Team],[Efficient_Shooting],0.266667,0.888889,1.777778
16,"[Winning_Team, Efficient_Shooting]",[High_Offense],0.266667,1.000000,2.000000



Association Rules: Confidence


,antecedents,consequents,confidence,support,lift
6,[Winning_Team],[High_Offense],1.000000,0.300000,2.000000
26,"[Winning_Team, Elite_Defense, Efficient_Shooting]",[High_Offense],1.000000,0.200000,2.000000
9,"[Elite_Defense, Efficient_Shooting]",[High_Offense],1.000000,0.300000,2.000000
16,"[Winning_Team, Efficient_Shooting]",[High_Offense],1.000000,0.266667,2.000000
22,"[Elite_Defense, Winning_Team]",[High_Offense],1.000000,0.233333,2.000000
17,"[High_Offense, Winning_Team]",[Efficient_Shooting],0.888889,0.266667,1.777778
18,[Winning_Team],"[High_Offense, Efficient_Shooting]",0.888889,0.266667,2.051282
2,[Winning_Team],[Efficient_Shooting],0.888889,0.266667,1.777778
0,[High_Offense],[Efficient_Shooting],0.866667,0.433333,1.733333
1,[Efficient_Shooting],[High_Offense],0.866667,0.433333,1.733333



Association Rules: LIft


,antecedents,consequents,lift,support,confidence
13,[Winning_Team],"[Elite_Defense, Efficient_Shooting]",2.222222,0.200000,0.666667
28,"[Elite_Defense, Efficient_Shooting]","[High_Offense, Winning_Team]",2.222222,0.200000,0.666667
10,"[Elite_Defense, Efficient_Shooting]",[Winning_Team],2.222222,0.200000,0.666667
32,[Winning_Team],"[High_Offense, Elite_Defense, Efficient_Shooting]",2.222222,0.200000,0.666667
24,"[High_Offense, Elite_Defense, Efficient_Shooting]",[Winning_Team],2.222222,0.200000,0.666667
30,"[High_Offense, Winning_Team]","[Elite_Defense, Efficient_Shooting]",2.222222,0.200000,0.666667
23,[Winning_Team],"[High_Offense, Elite_Defense]",2.121212,0.233333,0.777778
18,[Winning_Team],"[High_Offense, Efficient_Shooting]",2.051282,0.266667,0.888889
29,"[Winning_Team, Efficient_Shooting]","[High_Offense, Elite_Defense]",2.045455,0.200000,0.750000
9,"[Elite_Defense, Efficient_Shooting]",[High_Offense],2.000000,0.300000,1.000000
